In [2]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
import joblib

# 1. Load ONLY the two undersampled files
X = pd.read_csv('undersampled_features.csv').fillna(0)
y = pd.read_csv('undersampled_target.csv').fillna(0).values.ravel()

# 2. Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 3. Choose k using only the training part (5-fold cross-validation)
search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid={'n_neighbors': [9, 15, 21, 31, 51, 71, 101]},
    scoring='roc_auc',
    cv=StratifiedKFold(5, shuffle=True, random_state=42))
search.fit(X_train, y_train)
print("Best k:", search.best_params_['n_neighbors'], "| CV ROC-AUC:", round(search.best_score_, 3))

# 4. Evaluate once on the 20% test part
best = search.best_estimator_
pred = best.predict(X_test)
proba = best.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred)) #testing accuracy
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("PR-AUC:", round(average_precision_score(y_test, proba), 3))
print(confusion_matrix(y_test, pred))

#Cross Validation matrices
#import numpy as np
#from sklearn.model_selection import cross_validate

#cv_res = cross_validate(
    #search.best_estimator_, X_train, y_train,
    #cv=StratifiedKFold(5, shuffle=True, random_state=42),
    #scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision'])

#for name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision']:
    #s = cv_res['test_' + name]
    #print(f"{name:18s} {s.mean():.3f} ± {s.std():.3f}")

# 5. Save the model
#joblib.dump(best, 'my_knn_model.pkl')
#print("Model saved")

Best k: 101 | CV ROC-AUC: 0.819
              precision    recall  f1-score   support

         0.0       0.77      0.69      0.73      5635
         1.0       0.72      0.80      0.76      5676

    accuracy                           0.74     11311
   macro avg       0.75      0.74      0.74     11311
weighted avg       0.75      0.74      0.74     11311

ROC-AUC: 0.819
PR-AUC: 0.792
[[3896 1739]
 [1146 4530]]
accuracy           0.743 ± 0.004
precision          0.720 ± 0.007
recall             0.796 ± 0.006
f1                 0.756 ± 0.003
roc_auc            0.819 ± 0.003
average_precision  0.792 ± 0.002
